In [ ]:
import pandas as pd

data_category = ['affected-data.category.Financial Data',
'affected-data.category.HR Data',
'affected-data.category.IP',
'affected-data.category.Medical Data',
'affected-data.category.PII'
]

employee = ['indirect-victim.Employee.count',
'indirect-victim.Employee.impact.Data Exposure',
'indirect-victim.Employee.impact.Data Theft',
'indirect-victim.Employee.impact.Service Disruption']

entity = [
'indirect-victim.Entity.impact.Data Exposure',
'indirect-victim.Entity.impact.Data Loss',
'indirect-victim.Entity.impact.Data Theft']

# entity = ['indirect-victim.Entity.count',
# 'indirect-victim.Entity.impact.Data Exposure',
# 'indirect-victim.Entity.impact.Data Loss',
# 'indirect-victim.Entity.impact.Data Theft',
# 'indirect-victim.Entity.impact.Service Disruption',
# 'indirect-victim.Entity.impact.Service Diversion']

individual = ['indirect-victim.Individual.count',
'indirect-victim.Individual.impact.Data Exposure',
'indirect-victim.Individual.impact.Data Loss',
'indirect-victim.Individual.impact.Data Sale',
'indirect-victim.Individual.impact.Data Theft',
'indirect-victim.Individual.impact.Service Disruption',
'indirect-victim.Individual.impact.Service Diversion']

# select_cols= [*employee, *entity, *individual]
select_cols= [*entity]
entire_cols = ['uuid','victim','sector','sources',*select_cols]

# step 1 Only select victim records with associated indirect victims, 4070 records -> 774 

df =  pd.read_csv('incidents.csv', encoding="utf-8")

df[select_cols] = df[select_cols].replace({"": pd.NA,False:pd.NA})

df_filtered = df[df[select_cols].notna().any(axis=1)]

df_result = df_filtered[entire_cols]

df_result.to_csv("selected_sources.csv", index=False)

print(f"Original # of lines：{len(df)}")
print(f"After selected：{len(df_filtered)}\n")


Original # of lines：4070
After selected：336



In [ ]:
import requests
import ast
import pandas as pd

requests.packages.urllib3.disable_warnings()

# step 2 Filtered out sources that are no longer available, 5411 URLs -> , 774 records -> 


def is_url_alive(url: str, timeout=8) -> bool:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    try:
        resp = requests.get(
            url,
            headers=headers,
            timeout=timeout,
            verify=False,
            allow_redirects=True
        )
        return resp.status_code < 400
    except Exception as e:
        print(f"EXCEPTION for {url} | error: {repr(e)}")
        return False

df = pd.read_csv("selected_sources.csv")

keep_rows = []
number_urls = 0

for idx, row in df.iterrows():
    print("# of lines", idx+1)
    
    raw_src = row["sources"]

    try:
        url_list = ast.literal_eval(raw_src)
    except Exception:
        url_list = []

    alive_urls = [u for u in url_list if is_url_alive(u)]
    number_urls += len(url_list)
    #print("# of urls:", len(url_list), "alive:", len(alive_urls))

    if len(alive_urls) == 0:
        continue
    else:
        row["sources"] = alive_urls[0]
        keep_rows.append(row)
print("Total urls:", number_urls)

df_out = pd.DataFrame(keep_rows)
df_out.to_csv("filtered_sources.csv", index=False)

print(f"Original # of lines：{len(df)}")
print(f"After filtered：{len(df_out)}")

# of lines 1
EXCEPTION for http://www.topix.com/indy/2017/01/muncie-based-cancer-services-centers-servers-attacked-by-ransomware?fromrss=1 | error: ConnectionError(MaxRetryError('HTTPConnectionPool(host=\'www.topix.com\', port=80): Max retries exceeded with url: /indy/2017/01/muncie-based-cancer-services-centers-servers-attacked-by-ransomware?fromrss=1 (Caused by NameResolutionError("HTTPConnection(host=\'www.topix.com\', port=80): Failed to resolve \'www.topix.com\' ([Errno 11001] getaddrinfo failed)"))'))
EXCEPTION for http://www.topix.com/city/muncie-in/2017/01/indiana-cancer-agency-suffers-from-the-ransomware-attack?fromrss=1 | error: ConnectionError(MaxRetryError('HTTPConnectionPool(host=\'www.topix.com\', port=80): Max retries exceeded with url: /city/muncie-in/2017/01/indiana-cancer-agency-suffers-from-the-ransomware-attack?fromrss=1 (Caused by NameResolutionError("HTTPConnection(host=\'www.topix.com\', port=80): Failed to resolve \'www.topix.com\' ([Errno 11001] getaddrinfo fai

KeyboardInterrupt: 